# Trabalho de Fundamentos de Aprendizagem de Máquina
## Aprendizagem Supervisionada — Regressão e Classificação

**Dataset:** *Wine Quality* (Red Wine) — 1599 amostras, 11 atributos físico-químicos e a nota de qualidade (`quality`, inteiro entre 3 e 8).

Este notebook está dividido em duas partes:

- **Parte 1 — Regressão:** prever o valor *contínuo* da qualidade do vinho a partir das suas características.
- **Parte 2 — Classificação:** classificar o vinho em uma das 6 categorias de nota (3, 4, 5, 6, 7, 8).

Para cada parte, vários modelos são treinados, comparados via métricas quantitativas e o melhor é selecionado com justificativa.

## 1. Importação de bibliotecas e carga dos dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, StratifiedKFold, KFold)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Regressão ──────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Classificação ──────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42

In [ ]:
# Carga do dataset (mesmo arquivo disponibilizado na página principal da disciplina)
df = pd.read_csv("Dataset_Trabalho_Aprendizagem_Supervisionada.csv")
print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Valores nulos por coluna:")
print(df.isnull().sum())
print(f"\nLinhas duplicadas: {df.duplicated().sum()}")
print("  → As 240 duplicatas são mantidas por ser um dataset de domínio público amplamente")
print("    utilizado desta forma. Para fins acadêmicos comparativos, a remoção é opcional.")

## 2. Análise Exploratória dos Dados (EDA)

Antes de modelar, entendemos a distribuição da variável alvo e as relações entre as variáveis.

In [ ]:
# Distribuição da variável alvo
ax = sns.countplot(x="quality", data=df, palette="viridis")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.title("Distribuição das notas de qualidade (quality)")
plt.show()
print(df["quality"].value_counts().sort_index())

O dataset é **fortemente desbalanceado**: notas 5 e 6 representam ~82% das amostras,
enquanto as notas extremas (3, 4 e 8) são muito raras. Isso terá impacto direto na
**Parte 2 (Classificação)** — é fundamental usar `stratify` no split e métricas
robustas ao desbalanceamento como F1-macro e F1-weighted.

In [ ]:
# Matriz de correlação
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlação")
plt.show()

In [ ]:
# Correlação de cada feature com a variável alvo
corr_target = df.corr(numeric_only=True)["quality"].drop("quality").sort_values()
colors = ["steelblue" if v >= 0 else "indianred" for v in corr_target]
corr_target.plot(kind="barh", color=colors)
plt.title("Correlação das features com quality")
plt.xlabel("Coef. de correlação de Pearson")
plt.show()
print(corr_target.sort_values(ascending=False))

**Principais insights da correlação:**

- `alcohol` é a variável **mais positivamente correlacionada** (+0.48): vinhos com mais álcool tendem a ter nota maior.
- `volatile acidity` é a **mais negativamente correlacionada** (-0.39): alta acidez volátil prejudica a qualidade.
- `sulphates` (+0.25) e `citric acid` (+0.23) também contribuem positivamente.
- A maioria das variáveis tem correlação linear fraca com `quality`, o que indica que **modelos não-lineares (árvores, ensembles)** devem superar os lineares — hipótese confirmada nos resultados.

In [ ]:
# Boxplots das variáveis mais correlacionadas com a nota
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.flat, ["alcohol", "volatile acidity", "sulphates", "citric acid"]):
    sns.boxplot(x="quality", y=col, data=df, ax=ax, palette="viridis")
    ax.set_title(f"{col} por quality")
plt.tight_layout()
plt.show()

## 3. Preparação dos dados

Split 80/20 treino/teste.
- **Regressão:** sem `stratify` (alvo contínuo).
- **Classificação:** com `stratify=y` para preservar a proporção das classes minoritárias no conjunto de teste.

In [ ]:
X = df.drop(columns=["quality"])
y = df["quality"]

# Split para regressão
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
# Split para classificação (estratificado)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Treino (regressão)     : {X_train_r.shape}")
print(f"Teste  (regressão)     : {X_test_r.shape}")
print(f"Treino (classificação) : {X_train_c.shape}")
print(f"Teste  (classificação) : {X_test_c.shape}")

---
# Parte 1 — Regressão

**Objetivo:** prever o valor contínuo de `quality` a partir das 11 características físico-químicas.

**Modelos avaliados:**
1. Regressão Linear (*baseline*)
2. Ridge (regularização L2)
3. Lasso (regularização L1)
4. K-Nearest Neighbors Regressor (KNN)
5. Decision Tree Regressor
6. Random Forest Regressor
7. Gradient Boosting Regressor
8. Support Vector Regressor — SVR (kernel RBF)

**Métricas utilizadas:**
- **MAE** (Mean Absolute Error) — erro absoluto médio; na mesma unidade do alvo.
- **RMSE** (Root Mean Squared Error) — penaliza erros maiores; também na unidade do alvo.
- **R²** — fração da variância de `quality` explicada pelo modelo (0 = nada, 1 = perfeito).
- **CV-R²** — R² médio em validação cruzada 5-fold no treino (mede capacidade de generalização).

Modelos sensíveis à escala (Linear, Ridge, Lasso, KNN, SVR) são encapsulados em `Pipeline` com `StandardScaler`.

In [ ]:
def avalia_regressor(nome, modelo, X_tr, X_te, y_tr, y_te, cv=5):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)
    mae   = mean_absolute_error(y_te, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_te, y_pred))
    r2    = r2_score(y_te, y_pred)
    cv_r2 = cross_val_score(
        modelo, X_tr, y_tr,
        cv=KFold(cv, shuffle=True, random_state=RANDOM_STATE),
        scoring="r2"
    ).mean()
    return {"Modelo": nome, "MAE": round(mae, 4), "RMSE": round(rmse, 4),
            "R²": round(r2, 4), "CV-R²": round(cv_r2, 4)}, y_pred


modelos_reg = {
    "Linear Regression": Pipeline([("sc", StandardScaler()), ("m", LinearRegression())]),
    "Ridge":             Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
    "Lasso":             Pipeline([("sc", StandardScaler()), ("m", Lasso(alpha=0.001, random_state=RANDOM_STATE))]),
    "KNN Regressor":     Pipeline([("sc", StandardScaler()), ("m", KNeighborsRegressor(n_neighbors=10))]),
    "Decision Tree":     DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest":     RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, max_depth=3,
                                                   learning_rate=0.05, random_state=RANDOM_STATE),
    "SVR (RBF)":         Pipeline([("sc", StandardScaler()), ("m", SVR(kernel="rbf", C=1.0))]),
}

resultados_reg, preds_reg = [], {}
for nome, m in modelos_reg.items():
    res, yp = avalia_regressor(nome, m, X_train_r, X_test_r, y_train_r, y_test_r)
    resultados_reg.append(res)
    preds_reg[nome] = yp
    print(f"{nome:22s}  R²={res['R²']:.4f}  RMSE={res['RMSE']:.4f}  "
          f"MAE={res['MAE']:.4f}  CV-R²={res['CV-R²']:.4f}")

df_reg = pd.DataFrame(resultados_reg).sort_values("R²", ascending=False).reset_index(drop=True)
print("\n")
display(df_reg)

In [ ]:
# Visualização comparativa — Regressão
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_s1 = df_reg.sort_values("R²")
axes[0].barh(df_s1["Modelo"], df_s1["R²"], color="steelblue")
axes[0].set_title("R² no teste  (↑ melhor)")
axes[0].set_xlabel("R²")

df_s2 = df_reg.sort_values("RMSE", ascending=False)
axes[1].barh(df_s2["Modelo"], df_s2["RMSE"], color="indianred")
axes[1].set_title("RMSE no teste  (↓ melhor)")
axes[1].set_xlabel("RMSE")

plt.tight_layout()
plt.show()

### Observação — Decision Tree

O `DecisionTreeRegressor` obteve o **pior resultado** (R² ≈ 0.19, RMSE ≈ 0.73), inclusive abaixo dos modelos lineares.
O motivo é overfitting: com `max_depth=8` a árvore memoriza padrões do treino que não generalizam.
Ensembles como Random Forest corrigem isso ao combinar centenas de árvores rasas com bootstrap + seleção aleatória de features.

### Observação — SVR surpreende

O SVR com kernel RBF ficou em **2º lugar** (R² ≈ 0.46), superando até o Gradient Boosting isolado.
Isso reflete sua capacidade de modelar fronteiras não-lineares no espaço de features normalizado, sendo um forte candidato alternativo ao RF neste dataset.

### 3.1 Refinamento do melhor modelo — GridSearchCV no Random Forest

In [ ]:
# Espaço de busca reduzido para execução eficiente
param_grid_r = {
    "n_estimators":      [300, 600],
    "max_depth":         [None, 20],
    "min_samples_split": [2, 5],
}

gs_r = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_r,
    cv=5, scoring="r2", n_jobs=-1, verbose=1
)
gs_r.fit(X_train_r, y_train_r)

rf_best_r = gs_r.best_estimator_
y_pred_rf  = rf_best_r.predict(X_test_r)

print(f"Melhores parâmetros : {gs_r.best_params_}")
print(f"CV-R²  (treino)     : {gs_r.best_score_:.4f}")
print(f"R²     (teste)      : {r2_score(y_test_r, y_pred_rf):.4f}")
print(f"RMSE   (teste)      : {np.sqrt(mean_squared_error(y_test_r, y_pred_rf)):.4f}")
print(f"MAE    (teste)      : {mean_absolute_error(y_test_r, y_pred_rf):.4f}")

In [ ]:
# Gráfico Real vs Previsto
plt.scatter(y_test_r, y_pred_rf, alpha=0.45, color="steelblue")
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()], "r--", lw=2)
plt.xlabel("Qualidade real")
plt.ylabel("Qualidade prevista")
plt.title("Random Forest Regressor — Real vs Previsto")
plt.show()

In [ ]:
# Distribuição dos resíduos
residuos = y_test_r - y_pred_rf
sns.histplot(residuos, bins=30, kde=True, color="steelblue")
plt.axvline(0, color="red", linestyle="--")
plt.title("Distribuição dos resíduos (Real − Previsto)")
plt.xlabel("Erro")
plt.show()
print(f"Média dos resíduos : {residuos.mean():.4f}  (próximo de 0 → sem viés sistemático)")
print(f"Desvio padrão      : {residuos.std():.4f}")

In [ ]:
# Importância das variáveis
imp_r = pd.Series(rf_best_r.feature_importances_, index=X.columns).sort_values()
imp_r.plot(kind="barh", color="seagreen")
plt.title("Importância das variáveis — Random Forest Regressor")
plt.xlabel("Importância média (impureza de Gini)")
plt.show()
print(imp_r.sort_values(ascending=False))

### 3.2 Conclusão da Parte 1 — Regressão

| Modelo | R² | RMSE | MAE | CV-R² |
|---|---|---|---|---|
| **Random Forest** | **0.5347** | **0.5514** | **0.4242** | **0.4458** |
| SVR (RBF) | 0.4623 | 0.5928 | 0.4535 | 0.3617 |
| Gradient Boosting | 0.4399 | 0.6050 | 0.4870 | 0.3878 |
| Linear Regression | 0.4032 | 0.6245 | 0.5035 | 0.3226 |
| Ridge | 0.4032 | 0.6245 | 0.5036 | 0.3227 |
| Lasso | 0.4025 | 0.6249 | 0.5040 | 0.3235 |
| KNN Regressor | 0.3849 | 0.6340 | 0.4978 | 0.2798 |
| Decision Tree | 0.1868 | 0.7290 | 0.5199 | 0.1137 |

**Modelo escolhido: Random Forest Regressor.**
Melhor R² (0.53), menor RMSE (0.55) e maior CV-R² (0.45), demonstrando boa generalização.
A variável `alcohol` é, de longe, a mais importante, seguida de `sulphates` e `volatile acidity` — consistente com a análise de correlação da EDA.

---
# Parte 2 — Classificação

**Objetivo:** classificar o vinho em uma das 6 categorias de qualidade (3, 4, 5, 6, 7, 8).

**Modelos avaliados:**
1. Logistic Regression (multinomial via `solver='lbfgs'`)
2. K-Nearest Neighbors (KNN)
3. Naive Bayes (Gaussian)
4. Decision Tree
5. Random Forest
6. Gradient Boosting
7. Support Vector Machine — SVM (kernel RBF)

**Métricas utilizadas:**
- **Acurácia** — proporção de acertos totais.
- **F1-macro** — média do F1 por classe sem ponderar pelo suporte; penaliza desempenho ruim em classes minoritárias.
- **F1-weighted** — média ponderada pelo suporte de cada classe; mais influenciada pelas classes dominantes (5 e 6).
- **CV-Accuracy** — validação cruzada estratificada 5-fold para medir generalização.

In [ ]:
def avalia_classificador(nome, modelo, X_tr, X_te, y_tr, y_te, cv=5):
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(X_te)
    acc   = accuracy_score(y_te, y_pred)
    f1_m  = f1_score(y_te, y_pred, average="macro",    zero_division=0)
    f1_w  = f1_score(y_te, y_pred, average="weighted", zero_division=0)
    cv_acc = cross_val_score(
        modelo, X_tr, y_tr,
        cv=StratifiedKFold(cv, shuffle=True, random_state=RANDOM_STATE),
        scoring="accuracy"
    ).mean()
    return {"Modelo": nome, "Accuracy": round(acc, 4),
            "F1-macro": round(f1_m, 4), "F1-weighted": round(f1_w, 4),
            "CV-Accuracy": round(cv_acc, 4)}, y_pred


# Nota: LogisticRegression no sklearn >= 1.5 não usa mais o parâmetro
# 'multi_class'; o comportamento multinomial é padrão com solver='lbfgs'.
modelos_clf = {
    "Logistic Regression": Pipeline([
        ("sc", StandardScaler()),
        ("m",  LogisticRegression(max_iter=1000, solver="lbfgs",
                                  random_state=RANDOM_STATE))
    ]),
    "KNN":                 Pipeline([
        ("sc", StandardScaler()),
        ("m",  KNeighborsClassifier(n_neighbors=15))
    ]),
    "Naive Bayes":         GaussianNB(),
    "Decision Tree":       DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    "Random Forest":       RandomForestClassifier(n_estimators=300,
                                                  random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=300, max_depth=3,
                                                      learning_rate=0.05,
                                                      random_state=RANDOM_STATE),
    "SVM (RBF)":           Pipeline([
        ("sc", StandardScaler()),
        ("m",  SVC(kernel="rbf", C=1.0, random_state=RANDOM_STATE))
    ]),
}

resultados_clf, preds_clf = [], {}
for nome, m in modelos_clf.items():
    res, yp = avalia_classificador(nome, m, X_train_c, X_test_c, y_train_c, y_test_c)
    resultados_clf.append(res)
    preds_clf[nome] = yp
    print(f"{nome:22s}  Acc={res['Accuracy']:.4f}  "
          f"F1m={res['F1-macro']:.4f}  F1w={res['F1-weighted']:.4f}  "
          f"CV-Acc={res['CV-Accuracy']:.4f}")

df_clf = pd.DataFrame(resultados_clf).sort_values("Accuracy", ascending=False).reset_index(drop=True)
print("\n")
display(df_clf)

In [ ]:
# Visualização comparativa — Classificação
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_s1 = df_clf.sort_values("Accuracy")
axes[0].barh(df_s1["Modelo"], df_s1["Accuracy"], color="steelblue")
axes[0].set_title("Acurácia no teste  (↑ melhor)")
axes[0].set_xlabel("Accuracy")

df_s2 = df_clf.sort_values("F1-macro")
axes[1].barh(df_s2["Modelo"], df_s2["F1-macro"], color="darkorange")
axes[1].set_title("F1-macro no teste  (↑ melhor, sensível ao desbalanceamento)")
axes[1].set_xlabel("F1-macro")

plt.tight_layout()
plt.show()

### 4.1 Refinamento do melhor modelo — GridSearchCV no Random Forest Classifier

In [ ]:
param_grid_c = {
    "n_estimators":      [300, 600],
    "max_depth":         [None, 20],
    "min_samples_split": [2, 5],
    "max_features":      ["sqrt", "log2"],
}

gs_c = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_c,
    cv=5, scoring="accuracy", n_jobs=-1, verbose=1
)
gs_c.fit(X_train_c, y_train_c)

rfc_best = gs_c.best_estimator_
y_pred_c  = rfc_best.predict(X_test_c)

print(f"Melhores parâmetros : {gs_c.best_params_}")
print(f"CV-Acc  (treino)    : {gs_c.best_score_:.4f}")
print(f"Accuracy (teste)    : {accuracy_score(y_test_c, y_pred_c):.4f}")
print(f"F1-macro (teste)    : {f1_score(y_test_c, y_pred_c, average='macro', zero_division=0):.4f}")
print(f"F1-weighted (teste) : {f1_score(y_test_c, y_pred_c, average='weighted', zero_division=0):.4f}")

In [ ]:
print(classification_report(y_test_c, y_pred_c, zero_division=0))

In [ ]:
# Matriz de confusão
labels = sorted(y.unique())
cm = confusion_matrix(y_test_c, y_pred_c, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão — Random Forest Classifier")
plt.show()

In [ ]:
# Importância das variáveis
imp_c = pd.Series(rfc_best.feature_importances_, index=X.columns).sort_values()
imp_c.plot(kind="barh", color="darkorange")
plt.title("Importância das variáveis — Random Forest Classifier")
plt.xlabel("Importância média (impureza de Gini)")
plt.show()
print(imp_c.sort_values(ascending=False))

### 4.2 Desbalanceamento de classes e `class_weight="balanced"`

A matriz de confusão evidencia o problema: as classes **3 e 4** são praticamente nunca preditas corretamente —
o modelo as "ignora" em favor das classes majoritárias 5 e 6. Isso explica o **F1-macro baixo** (~0.41)
mesmo com acurácia de ~0.68.

Uma estratégia simples para tratar isso é usar `class_weight="balanced"`, que penaliza mais os erros nas
classes com poucas amostras. O tradeoff é uma pequena queda de acurácia, mas um ganho real em F1-macro.

In [ ]:
rfc_bal = RandomForestClassifier(
    **{**gs_c.best_params_,
       "class_weight": "balanced",
       "random_state":  RANDOM_STATE,
       "n_jobs":        -1}
)
rfc_bal.fit(X_train_c, y_train_c)
y_pred_bal = rfc_bal.predict(X_test_c)

print("═══ Comparativo: RF padrão vs RF balanceado ═══")
print(f"{'':30s}  {'Padrão':>10}  {'Balanceado':>10}")
print(f"{'Accuracy':30s}  {accuracy_score(y_test_c, y_pred_c):>10.4f}  "
      f"{accuracy_score(y_test_c, y_pred_bal):>10.4f}")
print(f"{'F1-macro':30s}  {f1_score(y_test_c, y_pred_c, average='macro', zero_division=0):>10.4f}  "
      f"{f1_score(y_test_c, y_pred_bal, average='macro', zero_division=0):>10.4f}")
print(f"{'F1-weighted':30s}  {f1_score(y_test_c, y_pred_c, average='weighted', zero_division=0):>10.4f}  "
      f"{f1_score(y_test_c, y_pred_bal, average='weighted', zero_division=0):>10.4f}")
print()
print("── classification_report (balanceado) ──")
print(classification_report(y_test_c, y_pred_bal, zero_division=0))

---
## 5. Conclusão Geral

### Parte 1 — Regressão

| Modelo | R² | RMSE | MAE | CV-R² |
|---|---|---|---|---|
| **Random Forest ✓** | **0.5347** | **0.5514** | **0.4242** | **0.4458** |
| SVR (RBF) | 0.4623 | 0.5928 | 0.4535 | 0.3617 |
| Gradient Boosting | 0.4399 | 0.6050 | 0.4870 | 0.3878 |
| Linear Regression | 0.4032 | 0.6245 | 0.5035 | 0.3226 |
| Ridge | 0.4032 | 0.6245 | 0.5036 | 0.3227 |
| Lasso | 0.4025 | 0.6249 | 0.5040 | 0.3235 |
| KNN Regressor | 0.3849 | 0.6340 | 0.4978 | 0.2798 |
| Decision Tree | 0.1868 | 0.7290 | 0.5199 | 0.1137 |

**Modelo escolhido: Random Forest Regressor** — melhor R² (0.53), menor RMSE (0.55) e maior CV-R² (0.45).

---

### Parte 2 — Classificação

| Modelo | Accuracy | F1-macro | F1-weighted | CV-Accuracy |
|---|---|---|---|---|
| **Random Forest ✓** | **0.6844** | 0.4080 | **0.6691** | **0.6763** |
| Gradient Boosting | 0.6500 | **0.4146** | 0.6401 | 0.6372 |
| SVM (RBF) | 0.6250 | 0.2990 | 0.6023 | 0.6052 |
| Decision Tree | 0.5906 | 0.3733 | 0.5909 | 0.5801 |
| Logistic Regression | 0.5906 | 0.2776 | 0.5673 | 0.5989 |
| KNN | 0.5750 | 0.2716 | 0.5550 | 0.5731 |
| Naive Bayes | 0.5719 | 0.3243 | 0.5757 | 0.5473 |

**Modelo escolhido: Random Forest Classifier** — maior acurácia (0.68), maior F1-weighted (0.67) e maior CV-Accuracy (0.68), indicando consistência e boa generalização.

> **Nota:** O Gradient Boosting apresentou F1-macro ligeiramente superior (0.4146 vs 0.4080). Caso o critério de escolha priorize o desempenho nas classes minoritárias, ele seria o candidato preferencial. A escolha do RF se justifica pela liderança em todas as demais métricas e pelo menor custo computacional de inferência.

---

### Observações finais

- A variável **`alcohol`** é a mais importante em ambos os problemas — regressão e classificação.
- O dataset é **fortemente desbalanceado**: notas 3, 4 e 8 são raramente preditas corretamente por qualquer modelo.
- Modelos baseados em **ensembles de árvores** (Random Forest e Gradient Boosting) superam consistentemente modelos lineares e baseados em distância, capturando as relações não-lineares entre as features e a qualidade.
- Para trabalhos futuros, recomenda-se explorar: **SMOTE** (oversampling das classes minoritárias), redefinição como **problema binário** (bom ≥ 7 vs ruim < 7), e modelos como **XGBoost / LightGBM / CatBoost**.